In [ ]:
!pip install gradio -q

from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import numpy as np
import pandas as pd
import cv2
import gradio as gr
import tensorflow as tf
import matplotlib.cm as cm

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# =============================================================================
# 1. LOAD IMAGE MODELS
# =============================================================================
binary_model = load_model("/content/drive/MyDrive/Saved_Models/resnet_model.keras")
multi_model  = load_model("/content/drive/MyDrive/multi_Saved_Models/resnet_model.keras")
print("✅ Image models loaded")

BINARY_CLASSES = {0: "Normal", 1: "Osteoporosis"}
MULTI_CLASSES  = {0: "Normal", 1: "Osteopenia", 2: "Osteoporosis"}

# =============================================================================
# 2. LOAD / TRAIN CSV MODEL
# =============================================================================
def train_csv_model():
    df = pd.read_csv('/content/osteoporosis.csv')
    df = df.drop(columns="Id", errors='ignore')
    df = df.fillna("None")
    df['Age_Group'] = pd.cut(df['Age'], bins=[0,30,50,70,100], labels=[0,1,2,3]).astype(int)
    df['Risk_Score'] = (
        (df['Smoking']           == 'Yes').astype(int) +
        (df['Family History']    == 'Yes').astype(int) +
        (df['Prior Fractures']   == 'Yes').astype(int) +
        (df['Calcium Intake']    == 'Low').astype(int) +
        (df['Vitamin D Intake']  == 'Insufficient').astype(int) +
        (df['Physical Activity'] == 'Sedentary').astype(int) +
        (df['Body Weight']       == 'Underweight').astype(int)
    )
    le_dict = {}
    for col in df.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le
    X = df.drop(columns='Osteoporosis')
    y = df['Osteoporosis']
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=24, stratify=y)
    gbm = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                                      subsample=0.85, min_samples_leaf=10, random_state=24)
    gbm.fit(X_train, y_train)
    save = {"model": gbm, "le_dict": le_dict, "features": list(X.columns)}
    with open("/content/csv_model.pkl", "wb") as f:
        pickle.dump(save, f)
    return gbm, le_dict, list(X.columns)

if os.path.exists("/content/csv_model.pkl"):
    with open("/content/csv_model.pkl", "rb") as f:
        saved = pickle.load(f)
    csv_model, csv_le_dict, csv_features = saved["model"], saved["le_dict"], saved["features"]
    print("✅ CSV model loaded")
else:
    csv_model, csv_le_dict, csv_features = train_csv_model()
    print("✅ CSV model trained")

# =============================================================================
# 3. GRAD-CAM HELPERS
# =============================================================================
def get_last_conv_layer_name(model):
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            for sublayer in reversed(layer.layers):
                if isinstance(sublayer, tf.keras.layers.Conv2D):
                    return sublayer.name
    raise ValueError("No Conv2D layer found")

def generate_gradcam(model, img_array, last_conv_layer_name):
    base_model = next((l for l in model.layers if isinstance(l, tf.keras.Model)), None)
    target_layer = base_model.get_layer(last_conv_layer_name)
    base_grad_model = tf.keras.Model(inputs=base_model.input,
                                      outputs=[target_layer.output, base_model.output])
    post_layers = model.layers[1:]
    with tf.GradientTape() as tape:
        conv_out, base_out = base_grad_model(img_array)
        x = base_out
        for layer in post_layers:
            x = layer(x)
        preds = x
        loss = preds[:, 0] if preds.shape[-1] == 1 else preds[:, tf.argmax(preds[0])]
    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = conv_out[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(img_array, heatmap, alpha=0.4):
    img = np.uint8(img_array)
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_colored = np.uint8(255 * cm.jet(heatmap_uint8)[:,:,:3])
    return np.clip(heatmap_colored * alpha + img, 0, 255).astype("uint8")

binary_conv_layer = get_last_conv_layer_name(binary_model)
multi_conv_layer  = get_last_conv_layer_name(multi_model)

# =============================================================================
# 4. PREDICTION FUNCTIONS
# =============================================================================
def predict_csv(age, gender, hormonal_changes, family_history, race,
                body_weight, calcium_intake, vitamin_d, physical_activity,
                smoking, alcohol, medical_conditions, medications, prior_fractures):
    raw = {
        "Age": age, "Gender": gender, "Hormonal Changes": hormonal_changes,
        "Family History": family_history, "Race/Ethnicity": race,
        "Body Weight": body_weight, "Calcium Intake": calcium_intake,
        "Vitamin D Intake": vitamin_d, "Physical Activity": physical_activity,
        "Smoking": smoking, "Alcohol Consumption": alcohol or "None",
        "Medical Conditions": medical_conditions or "None",
        "Medications": medications or "None", "Prior Fractures": prior_fractures,
    }
    input_df = pd.DataFrame([raw])
    input_df['Age_Group'] = pd.cut(input_df['Age'], bins=[0,30,50,70,100], labels=[0,1,2,3]).astype(int)
    input_df['Risk_Score'] = (
        (input_df['Smoking']           == 'Yes').astype(int) +
        (input_df['Family History']    == 'Yes').astype(int) +
        (input_df['Prior Fractures']   == 'Yes').astype(int) +
        (input_df['Calcium Intake']    == 'Low').astype(int) +
        (input_df['Vitamin D Intake']  == 'Insufficient').astype(int) +
        (input_df['Physical Activity'] == 'Sedentary').astype(int) +
        (input_df['Body Weight']       == 'Underweight').astype(int)
    )
    for col, le in csv_le_dict.items():
        if col in input_df.columns:
            val = str(input_df[col].iloc[0])
            input_df[col] = le.transform([val if val in le.classes_ else le.classes_[0]])
    input_df = input_df[csv_features]
    prob       = csv_model.predict_proba(input_df)[0]
    prediction = csv_model.predict(input_df)[0]
    confidence = prob[prediction] * 100
    risk_score = int(input_df['Risk_Score'].iloc[0])

    if prediction == 1:
        verdict_html = f"""
<div style="background:linear-gradient(135deg,#2d0a0a,#4a1010);border:1px solid #c0392b;
border-radius:16px;padding:28px 32px;margin:8px 0;font-family:'DM Sans',sans-serif;">
  <div style="display:flex;align-items:center;gap:14px;margin-bottom:20px;">
    <div style="width:52px;height:52px;background:#c0392b;border-radius:50%;display:flex;
    align-items:center;justify-content:center;font-size:24px;flex-shrink:0;">⚠️</div>
    <div>
      <div style="font-size:11px;letter-spacing:3px;color:#e57373;text-transform:uppercase;
      font-weight:600;margin-bottom:3px;">Screening Result</div>
      <div style="font-size:22px;font-weight:700;color:#ff6b6b;">HIGH RISK — Osteoporosis Detected</div>
    </div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:20px;">
    <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:14px 18px;">
      <div style="font-size:10px;letter-spacing:2px;color:#aaa;text-transform:uppercase;margin-bottom:4px;">Confidence</div>
      <div style="font-size:26px;font-weight:700;color:#ff6b6b;">{confidence:.1f}<span style="font-size:14px;">%</span></div>
    </div>
    <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:14px 18px;">
      <div style="font-size:10px;letter-spacing:2px;color:#aaa;text-transform:uppercase;margin-bottom:4px;">Risk Factors</div>
      <div style="font-size:26px;font-weight:700;color:#ff6b6b;">{risk_score}<span style="font-size:14px;color:#aaa;"> / 7</span></div>
    </div>
  </div>
  <div style="background:rgba(255,255,255,0.04);border-radius:10px;padding:14px 18px;margin-bottom:18px;">
    <div style="font-size:10px;letter-spacing:2px;color:#aaa;text-transform:uppercase;margin-bottom:10px;">Probability Distribution</div>
    <div style="margin-bottom:8px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <span style="font-size:12px;color:#aaa;">No Osteoporosis</span>
        <span style="font-size:12px;color:#fff;font-weight:600;">{prob[0]*100:.1f}%</span>
      </div>
      <div style="background:rgba(255,255,255,0.1);border-radius:4px;height:6px;">
        <div style="background:#4caf50;width:{prob[0]*100:.1f}%;height:100%;border-radius:4px;"></div>
      </div>
    </div>
    <div>
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <span style="font-size:12px;color:#aaa;">Osteoporosis</span>
        <span style="font-size:12px;color:#ff6b6b;font-weight:600;">{prob[1]*100:.1f}%</span>
      </div>
      <div style="background:rgba(255,255,255,0.1);border-radius:4px;height:6px;">
        <div style="background:#c0392b;width:{prob[1]*100:.1f}%;height:100%;border-radius:4px;"></div>
      </div>
    </div>
  </div>
  <div style="border-top:1px solid rgba(255,255,255,0.08);padding-top:16px;">
    <div style="font-size:11px;letter-spacing:2px;color:#aaa;text-transform:uppercase;margin-bottom:10px;">Clinical Recommendations</div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;">
      <div style="background:rgba(255,255,255,0.04);border-radius:8px;padding:10px 12px;
      font-size:12px;color:#ddd;">🩺 Bone density scan (DEXA)</div>
      <div style="background:rgba(255,255,255,0.04);border-radius:8px;padding:10px 12px;
      font-size:12px;color:#ddd;">💊 Review calcium & vitamin D</div>
      <div style="background:rgba(255,255,255,0.04);border-radius:8px;padding:10px 12px;
      font-size:12px;color:#ddd;">🏋️ Weight-bearing exercise</div>
      <div style="background:rgba(255,255,255,0.04);border-radius:8px;padding:10px 12px;
      font-size:12px;color:#ddd;">👨‍⚕️ Consult a specialist</div>
    </div>
  </div>
  <div style="margin-top:14px;font-size:11px;color:#666;font-style:italic;">
    ⚠️ For research purposes only. Not a substitute for clinical diagnosis.
  </div>
</div>"""
    else:
        verdict_html = f"""
<div style="background:linear-gradient(135deg,#0a2d14,#0d3b1a);border:1px solid #27ae60;
border-radius:16px;padding:28px 32px;margin:8px 0;font-family:'DM Sans',sans-serif;">
  <div style="display:flex;align-items:center;gap:14px;margin-bottom:20px;">
    <div style="width:52px;height:52px;background:#27ae60;border-radius:50%;display:flex;
    align-items:center;justify-content:center;font-size:24px;flex-shrink:0;">✅</div>
    <div>
      <div style="font-size:11px;letter-spacing:3px;color:#81c784;text-transform:uppercase;
      font-weight:600;margin-bottom:3px;">Screening Result</div>
      <div style="font-size:22px;font-weight:700;color:#69f0ae;">LOW RISK — No Osteoporosis Detected</div>
    </div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:20px;">
    <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:14px 18px;">
      <div style="font-size:10px;letter-spacing:2px;color:#aaa;text-transform:uppercase;margin-bottom:4px;">Confidence</div>
      <div style="font-size:26px;font-weight:700;color:#69f0ae;">{confidence:.1f}<span style="font-size:14px;">%</span></div>
    </div>
    <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:14px 18px;">
      <div style="font-size:10px;letter-spacing:2px;color:#aaa;text-transform:uppercase;margin-bottom:4px;">Risk Factors</div>
      <div style="font-size:26px;font-weight:700;color:#69f0ae;">{risk_score}<span style="font-size:14px;color:#aaa;"> / 7</span></div>
    </div>
  </div>
  <div style="background:rgba(255,255,255,0.04);border-radius:10px;padding:14px 18px;margin-bottom:18px;">
    <div style="font-size:10px;letter-spacing:2px;color:#aaa;text-transform:uppercase;margin-bottom:10px;">Probability Distribution</div>
    <div style="margin-bottom:8px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <span style="font-size:12px;color:#aaa;">No Osteoporosis</span>
        <span style="font-size:12px;color:#69f0ae;font-weight:600;">{prob[0]*100:.1f}%</span>
      </div>
      <div style="background:rgba(255,255,255,0.1);border-radius:4px;height:6px;">
        <div style="background:#27ae60;width:{prob[0]*100:.1f}%;height:100%;border-radius:4px;"></div>
      </div>
    </div>
    <div>
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <span style="font-size:12px;color:#aaa;">Osteoporosis</span>
        <span style="font-size:12px;color:#aaa;font-weight:600;">{prob[1]*100:.1f}%</span>
      </div>
      <div style="background:rgba(255,255,255,0.1);border-radius:4px;height:6px;">
        <div style="background:#c0392b;width:{prob[1]*100:.1f}%;height:100%;border-radius:4px;"></div>
      </div>
    </div>
  </div>
  <div style="border-top:1px solid rgba(255,255,255,0.08);padding-top:16px;">
    <div style="font-size:11px;letter-spacing:2px;color:#aaa;text-transform:uppercase;margin-bottom:10px;">Preventive Care Tips</div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;">
      <div style="background:rgba(255,255,255,0.04);border-radius:8px;padding:10px 12px;
      font-size:12px;color:#ddd;">🥛 Maintain calcium & vitamin D</div>
      <div style="background:rgba(255,255,255,0.04);border-radius:8px;padding:10px 12px;
      font-size:12px;color:#ddd;">🏃 Stay physically active</div>
      <div style="background:rgba(255,255,255,0.04);border-radius:8px;padding:10px 12px;
      font-size:12px;color:#ddd;">🚭 Avoid smoking & alcohol</div>
      <div style="background:rgba(255,255,255,0.04);border-radius:8px;padding:10px 12px;
      font-size:12px;color:#ddd;">📅 Regular check-ups after 50</div>
    </div>
  </div>
  <div style="margin-top:14px;font-size:11px;color:#666;font-style:italic;">
    ⚠️ For research purposes only. Not a substitute for clinical diagnosis.
  </div>
</div>"""
    return verdict_html


def predict_image(input_image):
    if input_image is None:
        return None, None, "<p style='color:#aaa;padding:20px;'>⚠️ Please upload an X-ray image.</p>"

    img      = input_image.resize((224, 224))
    img_orig = np.array(img).astype("float32")

    img_binary   = resnet_preprocess(np.expand_dims(img_orig.copy(), axis=0))
    binary_prob  = float(binary_model.predict(img_binary, verbose=0)[0][0])
    binary_label = "Osteoporosis" if binary_prob > 0.5 else "Normal"
    binary_conf  = binary_prob if binary_label == "Osteoporosis" else 1 - binary_prob

    try:
        bh = generate_gradcam(binary_model, img_binary, binary_conv_layer)
        b_overlay = Image.fromarray(overlay_heatmap(img_orig.copy(), bh))
    except:
        b_overlay = Image.fromarray(np.uint8(img_orig))

    if binary_label == "Normal":
        risk_pct   = binary_prob * 100
        risk_color = "#69f0ae" if risk_pct < 30 else ("#ffd54f" if risk_pct < 60 else "#ff6b6b")
        risk_label = "Low Risk" if risk_pct < 30 else ("Moderate Risk" if risk_pct < 60 else "High Risk")
        risk_icon  = "🟢" if risk_pct < 30 else ("🟡" if risk_pct < 60 else "🔴")

        report = f"""
<div style="font-family:'DM Sans',sans-serif;color:#e0e0e0;">
  <div style="background:linear-gradient(135deg,#0a2d14,#0d3b1a);border:1px solid #27ae60;
  border-radius:16px;padding:24px 28px;margin-bottom:14px;">
    <div style="font-size:10px;letter-spacing:3px;color:#81c784;text-transform:uppercase;margin-bottom:6px;">Stage 1 — Binary Screening</div>
    <div style="font-size:20px;font-weight:700;color:#69f0ae;margin-bottom:16px;">✅ Normal — No Osteoporosis</div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;">
      <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:12px;">
        <div style="font-size:9px;letter-spacing:2px;color:#888;text-transform:uppercase;margin-bottom:4px;">Confidence</div>
        <div style="font-size:22px;font-weight:700;color:#fff;">{binary_conf:.1%}</div>
      </div>
      <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:12px;">
        <div style="font-size:9px;letter-spacing:2px;color:#888;text-transform:uppercase;margin-bottom:4px;">Risk Score</div>
        <div style="font-size:22px;font-weight:700;color:{risk_color};">{risk_pct:.1f}%</div>
      </div>
      <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:12px;">
        <div style="font-size:9px;letter-spacing:2px;color:#888;text-transform:uppercase;margin-bottom:4px;">Risk Level</div>
        <div style="font-size:15px;font-weight:700;color:{risk_color};">{risk_icon} {risk_label}</div>
      </div>
    </div>
  </div>
  <div style="background:rgba(255,255,255,0.03);border:1px solid rgba(255,255,255,0.08);
  border-radius:12px;padding:14px 18px;font-size:12px;color:#888;">
    No Stage 2 severity staging required. No osteoporosis indicators detected.
  </div>
  <div style="margin-top:12px;font-size:11px;color:#555;font-style:italic;">
    ⚠️ For research purposes only. Consult a medical professional for diagnosis.
  </div>
</div>"""
        return b_overlay, Image.fromarray(np.uint8(img_orig)), report

    # Stage 2
    img_multi     = resnet_preprocess(np.expand_dims(img_orig.copy(), axis=0))
    multi_probs   = multi_model.predict(img_multi, verbose=0)[0]
    severity_idx  = int(np.argmax(multi_probs))
    severity_label = MULTI_CLASSES[severity_idx]
    severity_conf  = float(multi_probs[severity_idx])

    sev_color = {"Normal":"#69f0ae","Osteopenia":"#ffd54f","Osteoporosis":"#ff6b6b"}.get(severity_label,"#fff")
    sev_icon  = {"Normal":"🟢","Osteopenia":"🟡","Osteoporosis":"🔴"}.get(severity_label,"⚪")

    try:
        mh = generate_gradcam(multi_model, img_multi, multi_conv_layer)
        m_overlay = Image.fromarray(overlay_heatmap(img_orig.copy(), mh))
    except:
        m_overlay = Image.fromarray(np.uint8(img_orig))

    report = f"""
<div style="font-family:'DM Sans',sans-serif;color:#e0e0e0;">
  <div style="background:linear-gradient(135deg,#1a1a0d,#2d2a0a);border:1px solid #f39c12;
  border-radius:16px;padding:22px 26px;margin-bottom:12px;">
    <div style="font-size:10px;letter-spacing:3px;color:#f9ca74;text-transform:uppercase;margin-bottom:5px;">Stage 1 — Binary Screening</div>
    <div style="display:flex;justify-content:space-between;align-items:center;">
      <span style="font-size:16px;font-weight:600;color:#ff6b6b;">⚠️ Osteoporosis Detected</span>
      <span style="background:rgba(192,57,43,0.2);border:1px solid #c0392b;border-radius:20px;
      padding:4px 12px;font-size:12px;color:#ff6b6b;">{binary_conf:.1%} confidence</span>
    </div>
  </div>
  <div style="background:linear-gradient(135deg,#2d0a0a,#4a1010);border:1px solid #c0392b;
  border-radius:16px;padding:22px 26px;margin-bottom:12px;">
    <div style="font-size:10px;letter-spacing:3px;color:#e57373;text-transform:uppercase;margin-bottom:5px;">Stage 2 — Severity Classification</div>
    <div style="font-size:20px;font-weight:700;color:{sev_color};margin-bottom:16px;">{sev_icon} {severity_label}</div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:16px;">
      <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:12px;">
        <div style="font-size:9px;letter-spacing:2px;color:#888;text-transform:uppercase;margin-bottom:4px;">Severity Confidence</div>
        <div style="font-size:24px;font-weight:700;color:{sev_color};">{severity_conf:.1%}</div>
      </div>
      <div style="background:rgba(255,255,255,0.05);border-radius:10px;padding:12px;">
        <div style="font-size:9px;letter-spacing:2px;color:#888;text-transform:uppercase;margin-bottom:4px;">Classification</div>
        <div style="font-size:15px;font-weight:700;color:{sev_color};">{sev_icon} {severity_label}</div>
      </div>
    </div>
    <div style="font-size:10px;letter-spacing:2px;color:#888;text-transform:uppercase;margin-bottom:10px;">Class Probabilities</div>
    {"".join([
      f'<div style="margin-bottom:8px;"><div style="display:flex;justify-content:space-between;margin-bottom:3px;">'
      f'<span style="font-size:12px;color:#aaa;">{MULTI_CLASSES[i]}</span>'
      f'<span style="font-size:12px;color:#fff;font-weight:600;">{multi_probs[i]:.1%}</span></div>'
      f'<div style="background:rgba(255,255,255,0.1);border-radius:4px;height:5px;">'
      f'<div style="background:{"#27ae60" if i==0 else "#f39c12" if i==1 else "#c0392b"};'
      f'width:{multi_probs[i]*100:.1f}%;height:100%;border-radius:4px;"></div></div></div>'
      for i in range(3)
    ])}
  </div>
  <div style="margin-top:8px;font-size:11px;color:#555;font-style:italic;">
    ⚠️ For research purposes only. Consult a medical professional for diagnosis.
  </div>
</div>"""
    return b_overlay, m_overlay, report


# =============================================================================
# 5. PROFESSIONAL CSS
# =============================================================================
PROFESSIONAL_CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&family=DM+Mono:wght@400;500&display=swap');

* { box-sizing: border-box; }

body, .gradio-container {
    font-family: 'DM Sans', sans-serif !important;
    background: #0d0d0f !important;
}

/* ── Global background ── */
.gradio-container {
    background: radial-gradient(ellipse at 20% 10%, #0d1a2e 0%, #0d0d0f 60%) !important;
    min-height: 100vh;
}

/* ── Hide footer ── */
footer, .footer { display: none !important; }

/* ── Header banner ── */
.app-header {
    background: linear-gradient(135deg, #0d1a2e 0%, #0a2640 50%, #0d1a2e 100%);
    border-bottom: 1px solid rgba(41,182,246,0.15);
    padding: 28px 40px;
    position: relative;
    overflow: hidden;
}
.app-header::before {
    content: '';
    position: absolute;
    top: -40px; right: -40px;
    width: 200px; height: 200px;
    background: radial-gradient(circle, rgba(41,182,246,0.08) 0%, transparent 70%);
    border-radius: 50%;
}

/* ── Tabs ── */
.tab-nav {
    background: rgba(255,255,255,0.02) !important;
    border-bottom: 1px solid rgba(255,255,255,0.07) !important;
    padding: 0 16px !important;
}
.tab-nav button {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 13px !important;
    font-weight: 500 !important;
    letter-spacing: 0.3px !important;
    color: #888 !important;
    padding: 14px 20px !important;
    border-radius: 0 !important;
    border-bottom: 2px solid transparent !important;
    transition: all 0.2s ease !important;
}
.tab-nav button.selected {
    color: #29b6f6 !important;
    border-bottom: 2px solid #29b6f6 !important;
    background: transparent !important;
}
.tab-nav button:hover:not(.selected) {
    color: #ccc !important;
    background: rgba(255,255,255,0.03) !important;
}

/* ── Section cards ── */
.section-card {
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(255,255,255,0.07);
    border-radius: 14px;
    padding: 20px 22px;
    margin-bottom: 4px;
}

/* ── Input labels ── */
label span, .label-wrap span {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    letter-spacing: 1.5px !important;
    text-transform: uppercase !important;
    color: #8a9bb0 !important;
}

/* ── Sliders ── */
input[type=range] { accent-color: #29b6f6 !important; }
.wrap.svelte-i3tvor { border-color: rgba(255,255,255,0.08) !important; }

/* ── Radio buttons ── */
.wrap.svelte-1p9xokt {
    background: rgba(255,255,255,0.03) !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    border-radius: 8px !important;
}
input[type=radio]:checked + span {
    color: #29b6f6 !important;
}

/* ── Dropdowns ── */
.wrap-inner.svelte-15lo0d8 {
    background: rgba(255,255,255,0.04) !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    border-radius: 8px !important;
    color: #e0e0e0 !important;
}

/* ── Primary predict button ── */
.predict-btn button, button.primary {
    background: linear-gradient(135deg, #0a4a7c, #29b6f6) !important;
    border: none !important;
    border-radius: 10px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-size: 14px !important;
    font-weight: 600 !important;
    letter-spacing: 0.5px !important;
    color: #fff !important;
    padding: 14px 28px !important;
    transition: all 0.2s ease !important;
    box-shadow: 0 4px 20px rgba(41,182,246,0.2) !important;
}
.predict-btn button:hover, button.primary:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 6px 28px rgba(41,182,246,0.35) !important;
}

/* ── Image upload ── */
.upload-container {
    border: 1.5px dashed rgba(41,182,246,0.3) !important;
    border-radius: 14px !important;
    background: rgba(41,182,246,0.03) !important;
    transition: all 0.2s !important;
}
.upload-container:hover {
    border-color: rgba(41,182,246,0.6) !important;
    background: rgba(41,182,246,0.06) !important;
}

/* ── Image output panels ── */
.image-output {
    border: 1px solid rgba(255,255,255,0.07) !important;
    border-radius: 12px !important;
    overflow: hidden !important;
}

/* ── Textbox / Markdown output areas ── */
.prose, .md {
    color: #d0d8e4 !important;
    font-family: 'DM Sans', sans-serif !important;
}

/* ── Examples table ── */
.examples-table {
    border: 1px solid rgba(255,255,255,0.07) !important;
    border-radius: 10px !important;
    overflow: hidden !important;
}
.examples-table tr:hover {
    background: rgba(41,182,246,0.06) !important;
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 5px; }
::-webkit-scrollbar-track { background: #0d0d0f; }
::-webkit-scrollbar-thumb { background: #2a3a4a; border-radius: 3px; }
"""

# =============================================================================
# 6. BUILD GRADIO APP
# =============================================================================
with gr.Blocks(
    title="OsteoScan AI — Clinical Risk Assessment",
    css=PROFESSIONAL_CSS,
) as demo:

    # ── Header ───────────────────────────────────────────────────────────────
    gr.HTML("""
    <div class="app-header">
      <div style="display:flex;align-items:center;gap:16px;margin-bottom:6px;">
        <div style="width:42px;height:42px;background:linear-gradient(135deg,#0a4a7c,#29b6f6);
        border-radius:10px;display:flex;align-items:center;justify-content:center;
        font-size:20px;flex-shrink:0;">🦴</div>
        <div>
          <div style="font-size:24px;font-weight:700;color:#fff;letter-spacing:-0.3px;">
            OsteoScan <span style="color:#29b6f6;">AI</span>
          </div>
          <div style="font-size:12px;color:#5a7a9a;letter-spacing:1px;text-transform:uppercase;
          font-weight:500;">Clinical Osteoporosis Risk Assessment System</div>
        </div>
        <div style="margin-left:auto;display:flex;gap:8px;">
          <div style="background:rgba(41,182,246,0.1);border:1px solid rgba(41,182,246,0.25);
          border-radius:20px;padding:5px 14px;font-size:11px;color:#29b6f6;font-weight:500;">
            GBM · 91%+ Accuracy
          </div>
          <div style="background:rgba(41,182,246,0.1);border:1px solid rgba(41,182,246,0.25);
          border-radius:20px;padding:5px 14px;font-size:11px;color:#29b6f6;font-weight:500;">
            ResNet50 · Two-Stage
          </div>
        </div>
      </div>
      <div style="font-size:13px;color:#5a7a9a;margin-top:4px;">
        Select an analysis mode below — lifestyle risk scoring or X-ray image diagnosis.
        &nbsp;·&nbsp; <span style="color:#3d5a70;font-style:italic;">Research use only. Not a clinical diagnostic tool.</span>
      </div>
    </div>
    """)

    # ── Tabs ─────────────────────────────────────────────────────────────────
    with gr.Tabs():

        # ══════════════════════════════════════════════════════════════════════
        # TAB 1 — LIFESTYLE / CSV
        # ══════════════════════════════════════════════════════════════════════
        with gr.TabItem("📋  Lifestyle Risk Assessment"):

            gr.HTML("""
            <div style="padding:18px 4px 8px;font-family:'DM Sans',sans-serif;">
              <div style="font-size:13px;color:#5a7a9a;line-height:1.6;">
                Enter the patient's clinical and lifestyle profile below.
                The <strong style="color:#29b6f6;">Gradient Boosting classifier</strong>
                will compute osteoporosis risk from 14 features including engineered
                age group buckets and a composite lifestyle risk score.
              </div>
            </div>
            """)

            with gr.Row(equal_height=False):

                # ── Col 1: Demographics ──────────────────────────────────────
                with gr.Column(scale=1):
                    gr.HTML('<div style="font-size:10px;letter-spacing:3px;color:#29b6f6;'
                            'text-transform:uppercase;font-weight:600;padding:4px 0 10px;">👤 Demographics</div>')
                    age = gr.Slider(18, 90, value=45, step=1, label="Age")
                    gender = gr.Radio(["Male","Female"], value="Female", label="Gender")
                    race = gr.Dropdown(["African American","Asian","Caucasian"],
                                       value="Caucasian", label="Race / Ethnicity")
                    hormonal_changes = gr.Radio(["Normal","Postmenopausal"],
                                                value="Normal", label="Hormonal Changes")
                    family_history = gr.Radio(["Yes","No"], value="No",
                                              label="Family History of Osteoporosis")

                # ── Col 2: Lifestyle ─────────────────────────────────────────
                with gr.Column(scale=1):
                    gr.HTML('<div style="font-size:10px;letter-spacing:3px;color:#29b6f6;'
                            'text-transform:uppercase;font-weight:600;padding:4px 0 10px;">🏃 Lifestyle Factors</div>')
                    body_weight = gr.Radio(["Normal","Underweight"], value="Normal", label="Body Weight")
                    physical_activity = gr.Radio(["Active","Sedentary"], value="Active",
                                                 label="Physical Activity")
                    calcium_intake = gr.Radio(["Adequate","Low"], value="Adequate",
                                              label="Calcium Intake")
                    vitamin_d = gr.Radio(["Sufficient","Insufficient"], value="Sufficient",
                                         label="Vitamin D Intake")
                    smoking = gr.Radio(["Yes","No"], value="No", label="Smoking")
                    alcohol = gr.Radio(["Moderate","None"], value="None",
                                       label="Alcohol Consumption")

                # ── Col 3: Medical + Examples ────────────────────────────────
                with gr.Column(scale=1):
                    gr.HTML('<div style="font-size:10px;letter-spacing:3px;color:#29b6f6;'
                            'text-transform:uppercase;font-weight:600;padding:4px 0 10px;">🏥 Medical History</div>')
                    medical_conditions = gr.Dropdown(
                        ["None","Hyperthyroidism","Rheumatoid Arthritis"],
                        value="None", label="Medical Conditions")
                    medications = gr.Dropdown(["None","Corticosteroids"],
                                              value="None", label="Current Medications")
                    prior_fractures = gr.Radio(["Yes","No"], value="No",
                                               label="Prior Fractures")

                    gr.HTML('<div style="font-size:10px;letter-spacing:3px;color:#29b6f6;'
                            'text-transform:uppercase;font-weight:600;padding:18px 0 10px;">💡 Sample Patients</div>')
                    gr.Examples(
                        examples=[
                            [69,"Female","Postmenopausal","Yes","Asian","Underweight","Low",
                             "Insufficient","Sedentary","Yes","Moderate","Rheumatoid Arthritis","Corticosteroids","Yes"],
                            [32,"Male","Normal","No","Caucasian","Normal","Adequate",
                             "Sufficient","Active","No","None","None","None","No"],
                            [55,"Female","Postmenopausal","Yes","Caucasian","Normal","Low",
                             "Insufficient","Sedentary","No","None","Hyperthyroidism","Corticosteroids","No"],
                        ],
                        inputs=[age, gender, hormonal_changes, family_history, race,
                                body_weight, calcium_intake, vitamin_d, physical_activity,
                                smoking, alcohol, medical_conditions, medications, prior_fractures],
                        label="",
                    )

            # ── Predict Button ───────────────────────────────────────────────
            gr.HTML('<div style="height:8px;"></div>')
            with gr.Row():
                csv_predict_btn = gr.Button(
                    "⚡  Run Lifestyle Risk Assessment",
                    variant="primary", size="lg", elem_classes="predict-btn"
                )

            csv_result = gr.HTML()

            csv_predict_btn.click(
                fn=predict_csv,
                inputs=[age, gender, hormonal_changes, family_history, race,
                        body_weight, calcium_intake, vitamin_d, physical_activity,
                        smoking, alcohol, medical_conditions, medications, prior_fractures],
                outputs=csv_result
            )

        # ══════════════════════════════════════════════════════════════════════
        # TAB 2 — X-RAY IMAGE
        # ══════════════════════════════════════════════════════════════════════
        with gr.TabItem("🩻  X-Ray Image Analysis"):

            gr.HTML("""
            <div style="padding:18px 4px 8px;font-family:'DM Sans',sans-serif;">
              <div style="font-size:13px;color:#5a7a9a;line-height:1.6;">
                Upload a bone X-ray image. The
                <strong style="color:#29b6f6;">two-stage ResNet50 pipeline</strong>
                first screens for osteoporosis presence, then classifies severity.
                Grad-CAM heatmaps highlight the bone regions driving each decision.
              </div>
              <div style="display:flex;gap:10px;margin-top:12px;">
                <div style="background:rgba(41,182,246,0.07);border:1px solid rgba(41,182,246,0.2);
                border-radius:8px;padding:8px 14px;font-size:12px;color:#5a8aaa;">
                  <strong style="color:#29b6f6;">Stage 1</strong> — Normal vs Osteoporosis
                </div>
                <div style="background:rgba(41,182,246,0.07);border:1px solid rgba(41,182,246,0.2);
                border-radius:8px;padding:8px 14px;font-size:12px;color:#5a8aaa;">
                  <strong style="color:#29b6f6;">Stage 2</strong> — Normal / Osteopenia / Osteoporosis
                </div>
                <div style="background:rgba(41,182,246,0.07);border:1px solid rgba(41,182,246,0.2);
                border-radius:8px;padding:8px 14px;font-size:12px;color:#5a8aaa;">
                  <strong style="color:#29b6f6;">Grad-CAM</strong> — Explainability heatmaps
                </div>
              </div>
            </div>
            """)

            with gr.Row():
                # Upload panel
                with gr.Column(scale=1):
                    gr.HTML('<div style="font-size:10px;letter-spacing:3px;color:#29b6f6;'
                            'text-transform:uppercase;font-weight:600;padding:4px 0 10px;">Upload X-Ray</div>')
                    input_img = gr.Image(
                        type="pil", label="",
                        elem_classes="upload-container",
                        height=280
                    )
                    gr.HTML('<div style="height:8px;"></div>')
                    img_predict_btn = gr.Button(
                        "🔬  Analyze X-Ray",
                        variant="primary", size="lg", elem_classes="predict-btn"
                    )

                # Results panel
                with gr.Column(scale=2):
                    gr.HTML('<div style="font-size:10px;letter-spacing:3px;color:#29b6f6;'
                            'text-transform:uppercase;font-weight:600;padding:4px 0 10px;">Grad-CAM Visualisation</div>')
                    with gr.Row():
                        with gr.Column():
                            gr.HTML('<div style="font-size:11px;color:#5a7a9a;margin-bottom:6px;'
                                    'font-weight:500;">Stage 1 — Binary</div>')
                            stage1_out = gr.Image(label="", elem_classes="image-output", height=220)
                        with gr.Column():
                            gr.HTML('<div style="font-size:11px;color:#5a7a9a;margin-bottom:6px;'
                                    'font-weight:500;">Stage 2 — Severity</div>')
                            stage2_out = gr.Image(label="", elem_classes="image-output", height=220)

                    gr.HTML('<div style="font-size:10px;letter-spacing:3px;color:#29b6f6;'
                            'text-transform:uppercase;font-weight:600;padding:14px 0 6px;">Analysis Report</div>')
                    img_result = gr.HTML()

            img_predict_btn.click(
                fn=predict_image,
                inputs=input_img,
                outputs=[stage1_out, stage2_out, img_result]
            )

        # ══════════════════════════════════════════════════════════════════════
        # TAB 3 — ABOUT
        # ══════════════════════════════════════════════════════════════════════
        with gr.TabItem("ℹ️  About"):
            gr.HTML("""
            <div style="font-family:'DM Sans',sans-serif;max-width:860px;padding:28px 4px;color:#c0ccd8;">

              <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:28px;">

                <div style="background:rgba(41,182,246,0.04);border:1px solid rgba(41,182,246,0.12);
                border-radius:14px;padding:22px 24px;">
                  <div style="font-size:10px;letter-spacing:3px;color:#29b6f6;text-transform:uppercase;
                  font-weight:600;margin-bottom:10px;">📋 Lifestyle Risk Module</div>
                  <div style="font-size:13px;line-height:1.7;color:#8a9bb0;margin-bottom:14px;">
                    Gradient Boosting Classifier trained on 1,958 patient records with 14 clinical
                    and lifestyle features. Engineered features include age-group buckets and a
                    composite 7-factor risk score.
                  </div>
                  <div style="display:flex;flex-wrap:wrap;gap:6px;">
                    <span style="background:rgba(41,182,246,0.08);border:1px solid rgba(41,182,246,0.2);
                    border-radius:20px;padding:3px 10px;font-size:11px;color:#29b6f6;">GBM</span>
                    <span style="background:rgba(41,182,246,0.08);border:1px solid rgba(41,182,246,0.2);
                    border-radius:20px;padding:3px 10px;font-size:11px;color:#29b6f6;">91%+ Accuracy</span>
                    <span style="background:rgba(41,182,246,0.08);border:1px solid rgba(41,182,246,0.2);
                    border-radius:20px;padding:3px 10px;font-size:11px;color:#29b6f6;">14 Features</span>
                    <span style="background:rgba(41,182,246,0.08);border:1px solid rgba(41,182,246,0.2);
                    border-radius:20px;padding:3px 10px;font-size:11px;color:#29b6f6;">Scikit-learn</span>
                  </div>
                </div>

                <div style="background:rgba(41,182,246,0.04);border:1px solid rgba(41,182,246,0.12);
                border-radius:14px;padding:22px 24px;">
                  <div style="font-size:10px;letter-spacing:3px;color:#29b6f6;text-transform:uppercase;
                  font-weight:600;margin-bottom:10px;">🩻 X-Ray Analysis Module</div>
                  <div style="font-size:13px;line-height:1.7;color:#8a9bb0;margin-bottom:14px;">
                    Two-stage ResNet50-based deep learning pipeline. Stage 1 binary screens for
                    osteoporosis presence; Stage 2 classifies severity into three classes.
                    Grad-CAM provides visual explainability.
                  </div>
                  <div style="display:flex;flex-wrap:wrap;gap:6px;">
                    <span style="background:rgba(41,182,246,0.08);border:1px solid rgba(41,182,246,0.2);
                    border-radius:20px;padding:3px 10px;font-size:11px;color:#29b6f6;">ResNet50</span>
                    <span style="background:rgba(41,182,246,0.08);border:1px solid rgba(41,182,246,0.2);
                    border-radius:20px;padding:3px 10px;font-size:11px;color:#29b6f6;">Two-Stage</span>
                    <span style="background:rgba(41,182,246,0.08);border:1px solid rgba(41,182,246,0.2);
                    border-radius:20px;padding:3px 10px;font-size:11px;color:#29b6f6;">Grad-CAM</span>
                    <span style="background:rgba(41,182,246,0.08);border:1px solid rgba(41,182,246,0.2);
                    border-radius:20px;padding:3px 10px;font-size:11px;color:#29b6f6;">TensorFlow</span>
                  </div>
                </div>
              </div>

              <div style="background:rgba(255,100,100,0.04);border:1px solid rgba(255,100,100,0.15);
              border-radius:12px;padding:16px 20px;font-size:12px;color:#8a6a6a;line-height:1.6;">
                <strong style="color:#e57373;">⚠️ Disclaimer:</strong> This system is developed for
                research and educational purposes only. It does not constitute medical advice and
                must not be used as a substitute for professional clinical diagnosis. Always consult
                a qualified healthcare professional for medical decisions.
              </div>
            </div>
            """)

# =============================================================================
# 7. LAUNCH
# =============================================================================
demo.launch(share=True, debug=True)

Mounted at /content/drive
✅ Image models loaded
✅ CSV model trained


/tmp/ipykernel_11070/4225543595.py:561: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b565be92a0542d49df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
